<a href="https://colab.research.google.com/github/BrunaFerreira/Mestrado_UNIFESP/blob/main/Revisao_1_Transformacoes_Lista_Artigos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Base Completa de Artigos**

In [1]:
!pip install Unidecode

In [2]:
import pandas as pd
import numpy as np
import re
from unidecode import unidecode
#from google.colab import drive
#drive.mount('/content/drive')

# **Step 1: Arquivos atualizados**

### **Base de Dados Scopus**

In [3]:
def clean_text(df, col):
    chars_to_remove = r"[.\-()/]"

    df[col] = (
        df[col]
        .astype(str)
        .map(unidecode)
        .str.upper()
        .str.replace(chars_to_remove, " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    return df

In [4]:
def read_file_scopus (path, file, query):
  path_file = path + file
  q  = pd.read_csv(path_file)
  q['Query'] = query
  q['Artigo'] = range(1,len(q)+1)
  q['Base'] = 'Scopus'
  q['Cod'] = '0'
  q = clean_text(q,'Title')
  q.sort_values('Title')

  return q

In [45]:
path = '/content/drive/MyDrive/0_Mestrado_unifesp/3_Pesquisa/Revisao/202607_Revisao_Completa/0_Abstracts_Arquivos/'
path_to = '/content/drive/MyDrive/0_Mestrado_unifesp/3_Pesquisa/Revisao/202607_Revisao_Completa/1_Artigos_Filtrados/'

In [6]:
scopus = pd.DataFrame()
for i in [6,7,9,11,12,13] :
  q = read_file_scopus(path, "20260710_Q"+ str(i) + "_Scopus.csv", i)
  scopus = pd.concat([scopus,q],ignore_index=True)
scopus['Codigo'] = (scopus.index + 1).astype(str).str.zfill(4)

In [7]:
## Quantidade de artigos na base Scopus
scopus.shape[0]

586

In [8]:
scopus.to_csv(path + 'papers_scopus_2018_Jun2026.csv', index=False)

### **Base de Dados PubMed**

In [9]:
def read_file_pubmed (path, query):
  path_file = path + "20260710_Q" + str(query) + "_Abs_PubMed.txt"

  # ler arquivo
  with open(path_file , "r", encoding="utf-8") as f:
      texto = f.read()

  # separar artigos
  papers = re.split(r'\n\s*(?=\d+\.\s[A-Z])', texto.strip())

  records = []

  for p in papers:

      id_match = re.match(r'(\d+)\.', p)
      doi_match = re.search(r'DOI:\s*(\S+)', p)
      pmid_match = re.search(r'PMID:\s*(\d+)', p)

      partes = re.split(r'\n\s*\n', p)

      # journal = partes[0] if len(partes) > 0 else None
      # title = partes[1] if len(partes) > 1 else None
      # authors = partes[2] if len(partes) > 2 else None

      abstract = None
      for t in partes:
          if len(t) > 400:  # geralmente o abstract é o bloco maior
              abstract = t
              break

      # year = None
      # if journal:
      #     y = re.search(r'(19|20)\d{2}', journal)
      #     if y:
      #         year = y.group()

      records.append({
          #"id": int(id_match.group(1)) if id_match else None,
          #"journal": journal,
          #"year": year,
          #"title": title,
          #"authors": authors,
          "abstract": abstract,
          "doi": doi_match.group(1) if doi_match else None,
          "pmid": pmid_match.group(1) if pmid_match else None
      })

  df = pd.DataFrame(records)

  df = df [['doi','abstract','pmid']]

  df['query'] = query
  df['artigo'] = range(1,len(df)+1)
  df['base'] = 'PubMed'
  df['cod'] = '1'

  return df

#### **Lista de Artigos**



In [10]:
def read_list_pubmed (path,query):
  path_file = path + "20260710_Q" + str(query) + "_List_PubMed.csv"
  q = pd.read_csv(path_file)

  q_list = q[['PMID', 'Title', 'Authors', 'Citation', 'First Author', 'Journal/Book',
         'Publication Year', 'Create Date', 'PMCID', 'NIHMS ID', 'DOI']]

  q_list.columns = ['pmid', 'title', 'authors', 'citation', 'first_author', 'journal_book',
         'publication_year', 'create_date', 'pmcid', 'nihms_id', 'doi']

  q_list['pmid'] = q_list['pmid'].astype(int)

  return q_list

#### **Lista de Abstract**

In [11]:
def read_abstract_pubmed (path,query):
  q_abstract = read_file_pubmed (path, query)
  q_abstract['pmid'] = q_abstract['pmid'].astype(int)
  return q_abstract

In [12]:
def pubmed_complete (path, query):
  q_list = read_list_pubmed (path,query)
  q_abstract = read_abstract_pubmed (path,query)
  q_pubmed = pd.merge(
    q_list,
    q_abstract,
    on=['pmid','doi'],
    how='left')
  q_pubmed['Title'] = q_pubmed['title']
  q_pubmed = q_pubmed.drop(columns=['title'])
  return q_pubmed


In [13]:
pubmed = pd.DataFrame()
for i in [6,7,9,11,12,13] :
  q = pubmed_complete(path, i)
  q = clean_text(q,'Title')
  q = q.sort_values('Title')
  pubmed = pd.concat([pubmed,q],ignore_index=True)

pubmed['Codigo'] = (pubmed.index + 1).astype(str).str.zfill(3)
pubmed['Codigo'] = '1' + pubmed['Codigo']

In [14]:
pubmed.loc[
    pubmed['Codigo'] == '1033',
    ['query', 'artigo', 'base']
] = [9, 24, 'Pubmed']

In [15]:
pubmed.to_csv(path + 'papers_pubmed_2018_Jun2026.csv', index=False)

In [16]:
### Quantidade de artigos Base Pubmed
pubmed.shape[0]

94

## **Base Completa: PubMed + Scopus**
### **Periodo de Referência: 01/2018 a 06/2026**

In [17]:
scopus['criterio'] = ''
scopus['Document Type'] = scopus['Document Type'].str.lower()

# Arquivos que são revisão de conferência não são considerados
scopus.loc[scopus['Document Type'] == 'conference review', ['criterio', 'status' ]] = ['Conference Review', '0']
scopus.loc[scopus['Document Type'] == 'review', 'criterio'] = 'Revisão'

papers_scopus = scopus[['DOI','Codigo','Query','Base','Artigo', 'Title', 'criterio']]
papers_scopus.columns = ['DOI','Codigo','Query','Base','Artigo','Nome','Criterios']

In [18]:
pubmed['criterio'] = ''
papers_pubmed = pubmed[['doi','Codigo','query','base','artigo', 'Title', 'criterio']]
papers_pubmed.columns = ['DOI','Codigo','Query','Base','Artigo','Nome','Criterios']

In [19]:
papers_revision = pd.concat([papers_scopus, papers_pubmed], ignore_index=True)
papers_revision['DOI'] = papers_revision['DOI'].fillna('-1').astype(str)
papers_revision = papers_revision.sort_values(by=['Nome', 'Query']).reset_index(drop=True)

In [20]:
#def reduce_text_name(df, doi_list):
#    mask = df['DOI'].isin(doi_list)
#
#    df.loc[mask, 'Nome'] = (
#        df.loc[mask, 'Nome']
#        .astype(str)
#        .str.slice(0, 25)
#    )
#
#    return df

In [21]:
#Transformando campo de titulo para casos em que o DOI é igual

#doi_list = [
#    '10.1016/j.compbiolchem.2026.109084',
#    '10.3390/s26103215',
#    '10.1038/s41746-024-01276-5',
#    '10.12688/f1000research.166307.3',
#    '10.11834/jig.250168',
#    '10.11897/SP.J.1016.2024.02678'
#]
#
#papers_revision_202606 = reduce_text_name(
#    papers_revision_202606,
#    doi_list
#)
#
#papers_revision_202606.loc[
#    papers_revision_202606['DOI'] == '10.11834/jig.250168',
#    'Nome'
#] = 'ADDRESSING FAIRNESS ISSUES IN DEEP LEARNING-BASED MEDICAL IMAGE ANALYSIS: A SYSTEMATIC REVIEW'

In [22]:
papers_revision['Codigo'] = '_' + papers_revision['Codigo']
# Ordenar base de dados por Base, sendo primeiro Scopus  e depois pubmed
# Ordenar em seguida por query e artigo
papers_revision = (
    papers_revision
    .sort_values(
        by=['Base', 'Query', 'Artigo'],
        ascending=[False, True, True]
    )
    .reset_index(drop=True)
)

papers_revision.to_csv(path_to + '2_Lista_Artigos_Novos_20260710.csv', index=False)

In [23]:
### Quantidade total de artigos Pubmed + Scopus sem filtro
papers_revision.shape[0]

680

# **Step 2: Filtros e Limpeza da Base de Dados**

## **Artigos da revisão anterior**

In [24]:
artigos_anteriores = pd.read_csv(path_to + '0_Lista_Artigos_Filtrados_Anotacoes_20260405.csv')
artigos_anteriores = artigos_anteriores[['DOI', 'Codigo', 'Query', 'Base', 'Artigo', 'Nome', 'Criterios',
       'Status', 'Repetidos']]
# Casos em que não são repetidos ou marcados como repetidos, recebe valor 0
artigos_anteriores['Repetidos'] = artigos_anteriores['Repetidos'].fillna(0).astype(int)
artigos_anteriores['DOI'] = artigos_anteriores['DOI'].fillna('-1').astype(str)
artigos_anteriores = clean_text(artigos_anteriores,'Nome')

### **Filtro de artigos duplicados**



In [25]:
### Quantidade de artigos anteriores
artigos_anteriores.shape[0]

610

In [26]:
### Quantidade de artigos duplicados
artigos_anteriores[artigos_anteriores['Repetidos'] == 1].shape[0]

164

In [27]:
'''
caminho = path_to + '0_Lista_Artigos_Filtrados_Anotacoes_20260405.csv'
# Artigos 2025
artigos_25 = pd.read_excel(caminho + '202512_Artigos.xlsx', dtype={"Codigo": str})
artigos_25['Nome'] = (
    artigos_25['Nome']
    .astype(str)
    #Remover espaço em branco no inicio e fim da coluna Nome
    .str.strip()
    #Remover parenteses e aspas
    .str.replace(r'[()"\']', '', regex=True)
    .str.upper()
)
#Substituir os espaços vazios da coluna "Status" por -1
artigos_25['Status'] = artigos_25['Status'].fillna(-1).astype(int)
artigos_25['Repetidos'] = (
    artigos_25['Critérios']
    # Transforma coluna Critérios em texto e maiuscula
    .astype(str)
    .str.upper()
    #Remover espaço em branco no inicio e fim da coluna
    .str.strip()
    .str.startswith('IGUAL')
    .astype(int)
)
artigos_25  = artigos_25.drop(artigos_25.columns[-2], axis = 1)
artigos_25.columns = ['Codigo','Status','Query','Base', 'Artigo','Nome','Criterios','Repetidos']
artigos_25.head()

'''

'\ncaminho = path_to + \'0_Lista_Artigos_Filtrados_Anotacoes_20260405.csv\'\n# Artigos 2025\nartigos_25 = pd.read_excel(caminho + \'202512_Artigos.xlsx\', dtype={"Codigo": str})\nartigos_25[\'Nome\'] = (\n    artigos_25[\'Nome\']\n    .astype(str)\n    #Remover espaço em branco no inicio e fim da coluna Nome\n    .str.strip()\n    #Remover parenteses e aspas\n    .str.replace(r\'[()"\']\', \'\', regex=True)\n    .str.upper()\n)\n#Substituir os espaços vazios da coluna "Status" por -1\nartigos_25[\'Status\'] = artigos_25[\'Status\'].fillna(-1).astype(int)\nartigos_25[\'Repetidos\'] = (\n    artigos_25[\'Critérios\']\n    # Transforma coluna Critérios em texto e maiuscula\n    .astype(str)\n    .str.upper()\n    #Remover espaço em branco no inicio e fim da coluna\n    .str.strip()\n    .str.startswith(\'IGUAL\')\n    .astype(int)\n)\nartigos_25  = artigos_25.drop(artigos_25.columns[-2], axis = 1)\nartigos_25.columns = [\'Codigo\',\'Status\',\'Query\',\'Base\', \'Artigo\',\'Nome\',\'Crite

In [28]:
artigos_anteriores.to_csv(path_to + '2_Lista_Artigos_Artigos_20260710.csv', index=False)

In [29]:
# Filtrar artigos únicos na lista antiga de revisão
# desconsiderar artigos repetidos
artigos_anteriores = clean_text(artigos_anteriores,'Nome')
artigos_anteriores_filter = artigos_anteriores[artigos_anteriores['Repetidos']==0]
artigos_anteriores_filter = artigos_anteriores_filter[['DOI','Nome','Criterios','Status','Repetidos']]
artigos_anteriores_filter = artigos_anteriores_filter.drop_duplicates(subset=['DOI','Nome'])

In [30]:
### Qtd de Artigos Antigos Unicos
artigos_anteriores_filter.shape[0]

415

In [31]:
artigos_anteriores_filter.to_csv(path_to + '3_Lista_Artigos_Anteriores_Sem_Duplicacoes_202607.csv', index=False)

## **Artigos Novos**

### **Filtro de artigos de 'Conference Review'**

In [32]:
papers_revision.Criterios.value_counts()

,count
Criterios,
,561
Conference Review,65
Revisão,54


In [33]:
papers_revision.loc[
    papers_revision['Criterios'].astype(str).str.strip().str.upper() == 'CONFERENCE REVIEW',
    'Status'
] = 0

In [34]:
papers_revision.Status.value_counts()

,count
Status,
0.0,65


###  **Filtro de artigos duplicados**

In [35]:
### Quantidade de artigos Novos
papers_revision.shape[0]

680

In [36]:
# Inicializa a coluna
papers_revision['Repetidos'] = 0

# DOI válido (diferente de -1)
mask_doi = papers_revision['DOI'] != '-1'

papers_revision.loc[mask_doi, 'Repetidos'] = (
    papers_revision.loc[mask_doi]
    .duplicated(subset=['DOI'], keep='first')
    .astype(int)
)

# DOI = -1 -> usa Nome
mask_nome = papers_revision['DOI'] == '-1'

papers_revision.loc[mask_nome, 'Repetidos'] = (
    papers_revision.loc[mask_nome]
    .duplicated(subset=['Nome'], keep='first')
    .astype(int)
)

In [37]:
papers_revision_filter = papers_revision[papers_revision['Repetidos']!=1]

In [38]:
### Quantidade de artigos Novos Validos
papers_revision_filter.shape[0]

461

In [39]:

papers_revision_filter.to_csv(path_to + '3_Lista_Artigos_Novos_Sem_Duplicacoes_202607.csv', index=False)

#**Step 3: Comparar Lista de Artigos de 2025 e Lista de Artigos 202607**

In [40]:
# Verificar se os arquivos da scopus estão na base de dados antiga
papers_revision_filter = papers_revision_filter.merge(
    artigos_anteriores_filter,
    on=['DOI', 'Nome'],
    how='left',
    suffixes=('', '_Anteriores')
)

In [41]:
def definir_criterio_final(row):
    criterio = str(row['Criterios']).strip() if pd.notna(row['Criterios']) else ''
    criterio_ant = str(row['Criterios_Anteriores']).strip() if pd.notna(row['Criterios_Anteriores']) else ''

    # Se forem iguais
    if criterio == criterio_ant:
        return criterio

    # Se uma das colunas estiver vazia
    if criterio == '' or criterio_ant == '':
        return criterio + criterio_ant

    # Se forem diferentes
    return criterio + ' | ' + criterio_ant

papers_revision_filter['Criterio_Final'] = papers_revision_filter.apply(definir_criterio_final, axis=1)

papers_revision_filter['Status_Final'] = papers_revision_filter[['Status', 'Status_Anteriores']].fillna(-1).max(axis=1)

papers_revision_filter  = papers_revision_filter[['DOI', 'Codigo', 'Query', 'Base', 'Artigo', 'Nome', 'Criterio_Final', 'Repetidos',   'Status_Final']]

### Base completa de artigos avaliados

In [42]:
resultado = papers_revision.merge(
    papers_revision_filter,
    on=['Codigo','Query','Base','Artigo','Nome'],
    how='left',
    suffixes=('', '_Anteriores')
)

resultado.loc[resultado['Repetidos'] == 1, 'Status_Final'] = 0

In [43]:
resultado  = resultado[['DOI', 'Codigo', 'Query', 'Base', 'Artigo', 'Nome',
           'Criterio_Final','Repetidos', 'Status_Final']]

In [44]:
resultado.Status_Final.value_counts()

,count
Status_Final,
0.0,482
1.0,149
-1.0,49


In [45]:
resultado.to_csv(path_to + '4_Lista_Intermediaria_Artigos_202607.csv', index=False)

# Base de Dados sem repetidos
resultado  = resultado[resultado['Repetidos']!=1]